In [2]:
import json
import pandas as pd 

with open('traces_progresso.json', 'r', encoding='utf-8') as f:
    progresso = json.load(f)

df_queries = pd.read_csv('queries_geradas.csv')

linhas = []
for chave, trace in progresso.items():
    if trace is None:          # pula as 34 que nunca foram geradas
        continue
    i = int(chave)             # chave do JSON e texto; indice do DataFrame e numero
    linhas.append({
        "indice": i,
        "dificuldade_query": df_queries.loc[i, "dificuldade_query"],
        "trace": trace,
    })

df = pd.DataFrame(linhas)
print(len(df), "traces")
print(df['dificuldade_query'].value_counts().to_string())

484 traces
dificuldade_query
facil       216
dificil     140
sem_tool    128


In [ ]:
INSTRUCOES = """<instrucoes>
Você é um assistente de IA com acesso a um conjunto de ferramentas.
Seu objetivo é responder às perguntas do usuário de forma correta e útil.

Como proceder:

1. Analise o pedido do usuário e entenda a intenção.
2. Decida se precisa de ferramenta:
   - Se você consegue responder com o que já sabe, responda direto.
   - Se precisa de informação externa ou de executar uma ação, use uma ferramenta.
3. Se for usar uma ferramenta, escolha a mais adequada entre as disponíveis
   e extraia os argumentos a partir do pedido do usuário.
4. Para chamar uma ferramenta, escreva a chamada dentro de <tool_call>.
   O conteúdo deve ser um objeto JSON:

   <tool_call>
   {"nome_tool": "nome_exato_da_ferramenta", "argumentos": {"parametro": "valor"}}
   </tool_call>

5. O resultado da ferramenta será devolvido a você dentro de <tool_result>.
   Interprete esse resultado para responder ao pedido original.
6. Sua resposta final ao usuário deve estar sempre dentro de <final_answer>,
   em linguagem natural. Se a ferramenta retornou erro, explique isso ao usuário.

   <final_answer>
   Sua resposta ao usuário.
   </final_answer>
</instrucoes>"""




In [6]:
def transformar_trace(trace):
    nova = []
    ultima = len(trace) - 1           # posicao da ultima mensagem (2 ou 4)

    for pos, msg in enumerate(trace):
        role = msg["role"]
        conteudo = msg["content"]

        if pos == 0:
            conteudo = INSTRUCOES + "\n\n<ferramentas>\n" + conteudo + "</ferramentas>"

        elif role == "assistant" and pos == ultima:
            if "<final_answer>" not in conteudo:
                conteudo = "<final_answer>\n" + conteudo + "\n</final_answer>"

        elif role == "assistant":
            if "<tool_call>" not in conteudo:
                conteudo = "<tool_call>\n" + conteudo + "\n</tool_call>"

        elif role == "user" and pos != 1:
            if "<tool_result>" not in conteudo:
                conteudo = "<tool_result>\n" + conteudo + "\n</tool_result>"

        nova.append({"role": role, "content": conteudo})

    return nova

df["trace_formatada"] = df["trace"].apply(transformar_trace)
print(df.columns.tolist())


['indice', 'dificuldade_query', 'trace', 'trace_formatada']


In [7]:
for tipo in ["facil", "dificil", "sem_tool"]:
    print("=" * 70)
    print(f"TIPO: {tipo}")
    exemplo = df[df["dificuldade_query"] == tipo].iloc[0]
    for msg in exemplo["trace_formatada"]:
        print(f"\n--- {msg['role']} ---")
        print(msg["content"][:400])


TIPO: facil

--- system ---
<instrucoes>
Você é um assistente de IA com acesso a um conjunto de ferramentas.
Seu objetivo é responder às perguntas do usuário de forma correta e útil.

Como proceder:

1. Analise o pedido do usuário e entenda a intenção.
2. Decida se precisa de ferramenta:
   - Se você consegue responder com o que já sabe, responda direto.
   - Se precisa de informação externa ou de executar uma ação, use uma 

--- user ---
Quero agendar uma reunião para amanhã às 10h com o João e a Maria.

--- assistant ---
<tool_call>
{"nome_tool": "schedule_meeting", "argumentos": {"title": "Reuni\u00e3o de Trabalho", "date": "amanh\u00e3 \u00e0s 10h", "participants": ["Jo\u00e3o", "Maria"]}}
</tool_call>

--- user ---
<tool_result>
{
  "meeting_id": 12345,
  "title": "Reunião de Trabalho",
  "date": "2024-09-17T10:00:00",
  "participants": [
    {"name": "João", "email": "joao@example.com", "status": "confirmado"},
    {"name": "Maria", "email": "maria@example.com", "status": "agenda

In [8]:
partes = {"treino": [], "validacao": [], "teste": []}

for tipo, grupo in df.groupby("dificuldade_query"):
    g = grupo.sample(frac=1, random_state=42)      # embaralha dentro do tipo
    n = len(g)
    n_treino = int(0.8 * n)
    n_valid = int(0.1 * n)

    partes["treino"].append(g[:n_treino])
    partes["validacao"].append(g[n_treino:n_treino + n_valid])
    partes["teste"].append(g[n_treino + n_valid:])

splits = {}
for nome, lista in partes.items():
    splits[nome] = pd.concat(lista).sample(frac=1, random_state=42)
    vc = splits[nome]["dificuldade_query"].value_counts().to_dict()
    print(f"{nome}: {len(splits[nome])} traces  {vc}")


treino: 386 traces  {'facil': 172, 'dificil': 112, 'sem_tool': 102}
validacao: 47 traces  {'facil': 21, 'dificil': 14, 'sem_tool': 12}
teste: 51 traces  {'facil': 23, 'sem_tool': 14, 'dificil': 14}


In [9]:
for nome, dados in splits.items():
    caminho = f"{nome}.jsonl"
    with open(caminho, "w", encoding="utf-8") as f:
        for trace in dados["trace_formatada"]:
            f.write(json.dumps({"messages": trace}, ensure_ascii=False) + "\n")
    print(f"{caminho}: {len(dados)} linhas")


treino.jsonl: 386 linhas
validacao.jsonl: 47 linhas
teste.jsonl: 51 linhas
